In [19]:
import warnings
warnings.filterwarnings("ignore", category=Warning)
import numpy as np
import pandas as pd
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn

In [20]:
#First step is to load the data into an oject and setup our X and y (not used in our learning)
wineData = load_wine()
print(wineData.DESCR)

X = pd.DataFrame(wineData.data, columns=wineData.feature_names)
y = pd.Series(wineData.target)
X.head()

#Using the shape command, you can investigate the dimensionality
print('Shape of Data (X):',wineData.data.shape)
#print('Shape of Target Data (y)', wineData.target.shape)

.. _wine_dataset:

Wine recognition dataset
------------------------

**Data Set Characteristics:**

    :Number of Instances: 178
    :Number of Attributes: 13 numeric, predictive attributes and the class
    :Attribute Information:
 		- Alcohol
 		- Malic acid
 		- Ash
		- Alcalinity of ash  
 		- Magnesium
		- Total phenols
 		- Flavanoids
 		- Nonflavanoid phenols
 		- Proanthocyanins
		- Color intensity
 		- Hue
 		- OD280/OD315 of diluted wines
 		- Proline

    - class:
            - class_0
            - class_1
            - class_2
		
    :Summary Statistics:
    
    ============================= ==== ===== ======= =====
                                   Min   Max   Mean     SD
    ============================= ==== ===== ======= =====
    Alcohol:                      11.0  14.8    13.0   0.8
    Malic Acid:                   0.74  5.80    2.34  1.12
    Ash:                          1.36  3.23    2.36  0.27
    Alcalinity of Ash:            10.6  30.0    19.5   3.3
    Ma

In [21]:
X.head()

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0


In [22]:
y.value_counts()

1    71
0    59
2    48
Name: count, dtype: int64

In [23]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [24]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [25]:
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train.values, dtype=torch.long)
y_test = torch.tensor(y_test.values, dtype=torch.long)

In [26]:
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

X_train: torch.Size([142, 13])
y_train: torch.Size([142])


In [27]:
feature_names = wineData.feature_names
class_names = wineData.target_names

In [28]:
from model import SimpleANN

In [29]:
model = SimpleANN()

print(model)

SimpleANN(
  (layer1): Linear(in_features=13, out_features=16, bias=True)
  (layer2): Linear(in_features=16, out_features=8, bias=True)
  (output): Linear(in_features=8, out_features=3, bias=True)
)


In [30]:
loss_fn = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01
)

In [31]:
epochs = 200

for epoch in range(epochs):

    # Forward pass
    outputs = model(X_train)

    # Calculate loss
    loss = loss_fn(outputs, y_train)

    # Clear old gradients
    optimizer.zero_grad()

    # Backpropagation
    loss.backward()

    # Update weights
    optimizer.step()

    if (epoch + 1) % 20 == 0:
        print(
            f"Epoch [{epoch+1}/{epochs}], "
            f"Loss: {loss.item():.4f}"
        )

Epoch [20/200], Loss: 0.2979
Epoch [40/200], Loss: 0.0201
Epoch [60/200], Loss: 0.0031
Epoch [80/200], Loss: 0.0013
Epoch [100/200], Loss: 0.0009
Epoch [120/200], Loss: 0.0006
Epoch [140/200], Loss: 0.0005
Epoch [160/200], Loss: 0.0004
Epoch [180/200], Loss: 0.0004
Epoch [200/200], Loss: 0.0003


In [32]:
class_names = wineData.target_names

In [33]:
from sklearn.metrics import accuracy_score, classification_report

model.eval()

with torch.no_grad():
    test_outputs = model(X_test)
    test_predictions = torch.argmax(test_outputs, dim=1)

y_true = y_test.numpy()
y_pred = test_predictions.numpy()

accuracy = accuracy_score(y_true, y_pred)

print(f"Test accuracy: {accuracy:.4f}")
print()
print(classification_report(
    y_true,
    y_pred,
    target_names=class_names
))

Test accuracy: 0.9722

              precision    recall  f1-score   support

     class_0       1.00      1.00      1.00        12
     class_1       0.93      1.00      0.97        14
     class_2       1.00      0.90      0.95        10

    accuracy                           0.97        36
   macro avg       0.98      0.97      0.97        36
weighted avg       0.97      0.97      0.97        36



In [34]:
import os
import joblib
# Create an artifacts folder
os.makedirs("artifacts", exist_ok=True)

In [35]:
# Save the trained model's parameters
torch.save(
    model.state_dict(),
    "artifacts/wine_ann.pth"
)

# Save the preprocessing scaler and metadata
joblib.dump(
    {
        "scaler": scaler,
        "feature_names": list(feature_names),
        "class_names": list(class_names)
    },
    "artifacts/preprocessing.joblib"
)

print("Model and preprocessing information saved!")


Model and preprocessing information saved!
